# Arbitrary-Bit Fault: Model and Validation

A new, genuinely general SDC fault type: flips a uniformly random bit in Bob's live key
buffer once per Cascade pass, independent of whether Cascade is currently correcting
anything. This is the complement to `reconciliation_state` (which can only ever touch a bit
Cascade is actively correcting) -- this one can touch *any* bit, which means it actually
needs the pass-survival dynamics `point_estimate_model` was built for.

**Prerequisite**: apply the two edits in `arbitrary_bit_fault_PATCH.py` to your actual
`qne/cascade/fault_injection.py` and `qne/cascade/reconciliation.py` before running this.

**No real-channel data exists yet for this fault type** -- it's brand new. This notebook
generates validation data locally (via `MockClassicalSession`, your real `Reconciliation`
class, real key pairs) so you have something to test the model against tonight. Treat this
as a first pass, not a substitute for eventually confirming on the real channel.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import comb, ceil

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.fault_injection import SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.validation_utils import wilson_ci, chi_square_goodness_of_fit, report_fit

RESULTS = PROJECT_DIR / "results"
key_pairs_df = pd.read_csv(str(RESULTS / "key_pairs_metadata.csv"))


In [5]:
# --- FABRIC connection + real key pair (same as in your TMR notebook) ---
import deploy_fabric as deploy
import json

SLICE_NAME = 'qfabric-bb84-2'
fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
assert alice_indices == bob_indices

meta = json.loads(META_PATH.read_text())
k_pe, real_qber = meta["k"], meta["qber"]

alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")

print(f"Connected. n_bits={alice_key.get_nr_bits()}, k={k_pe}, qber={real_qber:.4f}")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid
Connected. n_bits=3488, k=388, qber=0.0077


## The model: `point_estimate_model`, reused exactly, plus one wrapper

`point_estimate_model` already answers "does a fresh error injected at pass i survive
detection through the remaining passes" -- exactly what a newly-introduced random error
needs. The only addition: a uniformly random bit has some chance of landing on an
already-wrong bit instead of a correct one, in which case the flip *fixes* it rather than
breaking anything. Weight by `(1 - K_i/N)`, the probability the random position was NOT
already one of the background errors.

In [ ]:
def block_schedule(qber, M=4):
    def m_block(i, q):
        return ceil(0.73 / q) if i == 1 else 2 * m_block(i - 1, q)
    return [m_block(i, qber) for i in range(1, M + 1)]

def P_survive_point_estimate(K, N, m):
    K = int(round(K))
    pop, sample = N - 1, m - 1
    denom = comb(pop, sample)
    s = 0
    for i in range(0, sample + 1, 2):
        if i <= K and (sample - i) <= (pop - K):
            s += comb(K, i) * comb(pop - K, sample - i)
    return s / denom

def P_odd_point(K, N, m):
    K = int(round(K))
    denom = comb(N, m)
    s = 0
    for i in range(1, m + 1, 2):
        if i <= K and (m - i) <= (N - K):
            s += comb(K, i) * comb(N - K, m - i)
    return s / denom

def point_estimate_model(n_bits, qber, M=4):
    """Unmodified from tonight's reconciliation-model notebook -- reused as-is,
    exactly the case it was built for."""
    m_sizes = [min(m, n_bits) for m in block_schedule(qber, M)]
    K = n_bits * qber
    K_trace = [K]
    C_trace, survive = [], []
    for m in m_sizes:
        n_blocks = n_bits / m
        p_odd = P_odd_point(K, n_bits, m)
        C_i = n_blocks * p_odd
        C_trace.append(C_i)
        survive.append(P_survive_point_estimate(K, n_bits, m))
        K = max(0, K - C_i)
        K_trace.append(K)
    mismatch_from_pass = [float(np.prod(survive[i:])) for i in range(M)]
    return {"m_sizes": m_sizes, "C_trace": C_trace, "survive": survive,
            "K_trace": K_trace, "mismatch_from_pass": mismatch_from_pass}


def arbitrary_bit_fault_model(n_bits, qber, M=4):
    """point_estimate_model, unmodified, plus the (1 - K_i/N) accidental-fix
    correction. K_trace[i] is background errors remaining BEFORE pass i+1 --
    i.e. before the fault at pass i+1 could fire."""
    base = point_estimate_model(n_bits, qber, M)
    N = n_bits
    adjusted = [(1 - base["K_trace"][i] / N) * base["mismatch_from_pass"][i] for i in range(M)]
    return {**base, "mismatch_from_pass_adjusted": adjusted}


def marginal_prediction_arbitrary(model_result, M=4):
    """Unlike reconciliation_state (weighted by C_i, since firing chance scales
    with how many corrections happen at that pass), this fault fires with the
    SAME probability every pass regardless of what Cascade is doing -- so
    marginalizing over 'which pass did the one fault land in' is a plain
    average, not a C_i-weighted one."""
    return float(np.mean(model_result["mismatch_from_pass_adjusted"][:M]))


## Generate local validation data (real Reconciliation class, MockClassicalSession)

In [ ]:
ARBITRARY_BIT_PROBS = [1e-3, 5e-3, 0.01, 0.03, 0.05, 0.1]
N_RUNS = 60

rows = []
for _, krow in key_pairs_df.iterrows():
    key_idx, n_bits, qber = int(krow["index"]), int(krow["n_bits"]), float(krow["qber"])

    alice_path = RESULTS / f"alice_sifted_bits_key{key_idx}.json"
    bob_path = RESULTS / f"bob_sifted_bits_key{key_idx}.json"
    if not (alice_path.exists() and bob_path.exists()):
        print(f"key{key_idx}: sifted-bit files not found, skipping -- adjust paths if yours differ")
        continue
    alice_key, _ = key_from_sifted_json(str(alice_path), "alice_bits")
    bob_key, _ = key_from_sifted_json(str(bob_path), "bob_bits")

    for prob in ARBITRARY_BIT_PROBS:
        for run in range(N_RUNS):
            injector = SDCFaultInjector(arbitrary_bit_prob=prob, seed=1000 + run)
            session = MockClassicalSession(correct_key=alice_key)
            recon = Reconciliation(algorithm=ORIGINAL, classical_session=session, noisy_key=bob_key,
                                     estimated_bit_error_rate=qber, seed=run,
                                     correct_key=alice_key, fault_injector=injector)
            reconciled = recon.reconcile()

            n_faults_fired = sum(1 for f in injector.injected_faults if f[0] == "arbitrary_bit")
            keys_match = alice_key.nr_bits_different(reconciled) == 0

            rows.append({"key_index": key_idx, "n_bits": n_bits, "qber": qber,
                          "prob": prob, "run": run, "n_faults_fired": n_faults_fired,
                          "keys_match": keys_match})

arbitrary_bit_df = pd.DataFrame(rows)
arbitrary_bit_df.to_csv(str(RESULTS / "arbitrary_bit_mock_local.csv"), index=False)
print(f"generated {len(arbitrary_bit_df)} local trials")


## Filter to single-fault runs and test the model

In [ ]:
single_fault_df = arbitrary_bit_df[arbitrary_bit_df["n_faults_fired"] == 1].copy()
single_fault_df["mismatch"] = ~single_fault_df["keys_match"]
print(f"single-fault-fired rows: {len(single_fault_df)}/{len(arbitrary_bit_df)}")

def predictor(krow, prob):
    model_result = arbitrary_bit_fault_model(int(krow["n_bits"]), float(krow["qber"]))
    return marginal_prediction_arbitrary(model_result)

chi2_s, dof, p = chi_square_goodness_of_fit(single_fault_df, key_pairs_df, predictor, group_col="prob")
report_fit("Arbitrary-bit fault, point_estimate_model (reused) + accidental-fix correction", chi2_s, dof, p)


## Per-key breakdown + plot

In [ ]:
print(f"{'key':<5} {'qber':<10} {'prob':<8} {'n_trials':<10} {'observed':<10} {'predicted':<10}")
for key_idx in sorted(single_fault_df["key_index"].unique()):
    krow = key_pairs_df.loc[key_pairs_df["index"] == key_idx].iloc[0]
    sub = single_fault_df[single_fault_df["key_index"] == key_idx]
    for prob in sorted(sub["prob"].unique()):
        trials = sub[sub["prob"] == prob]
        observed = trials["mismatch"].mean()
        predicted = predictor(krow, prob)
        print(f"{key_idx:<5} {float(krow['qber']):<10.4f} {prob:<8} {len(trials):<10} "
              f"{observed:<10.3f} {predicted:<10.3f}")


In [ ]:
probs = sorted(single_fault_df["prob"].unique())
pooled_obs, pooled_lo, pooled_hi, mean_pred = [], [], [], []
for prob in probs:
    sub = single_fault_df[single_fault_df["prob"] == prob]
    p_hat, lo, hi = wilson_ci(sub["mismatch"].sum(), len(sub))
    pooled_obs.append(p_hat); pooled_lo.append(lo); pooled_hi.append(hi)
    key_preds = [predictor(key_pairs_df.loc[key_pairs_df["index"]==k].iloc[0], prob)
                 for k in sub["key_index"].unique()]
    mean_pred.append(np.mean(key_preds))

pooled_obs, pooled_lo, pooled_hi = np.array(pooled_obs), np.array(pooled_lo), np.array(pooled_hi)
fig, ax = plt.subplots(figsize=(7, 5))
ax.errorbar(probs, pooled_obs, yerr=[np.clip(pooled_obs-pooled_lo,0,None), np.clip(pooled_hi-pooled_obs,0,None)],
              fmt="o-", capsize=3, label="observed (local Mock, pooled)")
ax.plot(probs, mean_pred, "x--", color="tab:red", label="predicted", markersize=9)
ax.set_xscale("log"); ax.set_xlabel("arbitrary_bit_prob"); ax.set_ylabel("Mismatch rate")
ax.set_title("Arbitrary-bit fault: model vs. local Mock data")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(RESULTS / "fig_arbitrary_bit_model.png"), dpi=150)
plt.show()


## Reading this

- If this fits well: you have a working, validated model for a fault type that genuinely
  needed the survival dynamics -- confirming `point_estimate_model` really is the right tool
  when the fault can land anywhere, unlike `reconciliation_state`.
- **This is local Mock data, not real-channel data.** Before presenting this fault type
  anywhere, run a modest real-channel confirmation batch and check it against the same
  model -- the whole point of tonight's real-vs-mock equivalence testing was that Mock is a
  legitimate *stand-in for exploration*, not a substitute for the real result once you're
  ready to claim something.

In [7]:
import time

ARBITRARY_BIT_PROBS = [1e-3, 1e-2, 0.05, 0.1, 0.3]
N_RUNS_PER_PROB = 8  # keep modest -- each is a real FABRIC round-trip

rows = []
for prob in ARBITRARY_BIT_PROBS:
    for run in range(N_RUNS_PER_PROB):
        seed = 5000 + run
        bob_output = f"results/bob_arbfault_p{prob}_run{run}.json"
        bob_thread = bob.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/bob_cascade_driver.py "
            f"--key-json results/bob_sifted_bits.json --alice-key-json results/alice_sifted_bits.json "
            f"--host {bob_ip} --port 5200 --qber {real_qber} --k {k_pe} --seed {seed} "
            f"--arbitrary-bit-prob {prob} --output {bob_output}"
        )
        alice_thread = alice.execute_thread(
            f"cd ~/qfabric && ~/qfabric/.venv/bin/python3 scripts/alice_cascade_responder.py "
            f"--key-json results/alice_sifted_bits.json --bob-host {bob_ip} --port 5200 "
            f"--seed {seed} --output results/alice_arbfault_p{prob}_run{run}.json"
        )
        bob_out = bob_thread.result(timeout=180)
        try: alice_thread.result(timeout=90)
        except Exception: pass

        stdout, _ = bob.execute(f"cat ~/qfabric/{bob_output}", quiet=True)
        try:
            data = json.loads(stdout)
            n_fired = data.get("faults_fired", {}).get("arbitrary_bit", 0)
            keys_match = data.get("remaining_errors_after_reconciliation", 1) == 0
        except (json.JSONDecodeError, KeyError):
            n_fired, keys_match = None, None
        rows.append({"key_index": 0, "prob": prob, "run": run, "n_faults_fired": n_fired, "keys_match": keys_match})
        time.sleep(2)

df_real_arbitrary = pd.DataFrame(rows)
df_real_arbitrary.to_csv(str(RESULTS / "arbitrary_bit_realchannel.csv"), index=False)

KeyboardInterrupt: 